# SmartCart Customer Segmentation System

**Minor Project | Unsupervised Machine Learning | Customer Analytics**

This notebook explores customer profiles from SmartCart using data cleaning, feature engineering, correlation analysis, encoding, scaling, PCA, K-Means, and Agglomerative Clustering. It concludes with cluster profiling intended to support more focused marketing and customer-engagement strategies.

> **Dataset note:** The supplied dataset contains 2,240 customer records and 22 columns. `Response` and `Complain` are available in the data, but the clustering pipeline treats segmentation as an unsupervised task and does not use either field as a supervised target. The resulting groups are exploratory profiles, not validated churn predictions.

## Notebook workflow
1. Load data and inspect data quality
2. Impute missing income values
3. Engineer age, tenure, spending, and household features
4. Remove identifier/redundant columns and inspect outliers
5. Explore numeric correlations
6. Encode categorical variables and scale features
7. Use PCA for visualization and clustering representation
8. Compare candidate K values with Elbow and Silhouette methods
9. Fit K-Means and Agglomerative models
10. Profile clusters and export useful tables/figures

## 1. Imports and configuration

In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score, adjusted_rand_score

RANDOM_STATE = 42
# Resolve output/data paths whether Jupyter starts in repo root or notebooks/.
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data" / "smartcart_customers.csv").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
DATA_PATH = PROJECT_ROOT / "data" / "smartcart_customers.csv"
FIGURES_DIR = PROJECT_ROOT / "outputs" / "figures"
TABLES_DIR = PROJECT_ROOT / "outputs" / "tables"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 140)

## 2. Load and understand the dataset
Each row represents a customer. Start by checking dimensions, types, sample rows, missing values, and duplicated records before modifying anything.

In [ ]:
df = pd.read_csv(DATA_PATH)

print(f"Dataset shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
display(df.head())
display(df.dtypes.rename("data_type").to_frame())

quality_summary = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_percent": (df.isna().mean() * 100).round(2),
    "unique_values": df.nunique(dropna=False)
}).sort_values("missing_count", ascending=False)
display(quality_summary)
print(f"Duplicate rows: {df.duplicated().sum()}")

## 3. Handle missing values
The supplied code fills missing `Income` entries with the median. Median imputation is less sensitive to extreme income values than mean imputation. Save the original missingness count so the cleaning step is auditable.

In [ ]:
df = df.copy()
income_missing_before = int(df["Income"].isna().sum())
income_median = df["Income"].median()
df["Income"] = df["Income"].fillna(income_median)
print(f"Income values imputed: {income_missing_before}")
print(f"Median used: {income_median:,.2f}")
assert df["Income"].isna().sum() == 0

## 4. Feature engineering
Create interpretable customer-level variables. Age uses a configurable reference year. Tenure is measured relative to the latest enrollment date in the supplied data, rather than the runtime date, so the result is reproducible.

In [ ]:
REFERENCE_YEAR = 2026

# Parse enrollment date explicitly using the day-first format shown in the source data.
df["Dt_Customer"] = pd.to_datetime(df["Dt_Customer"], dayfirst=True, errors="coerce")
if df["Dt_Customer"].isna().any():
    raise ValueError("Some enrollment dates could not be parsed. Inspect Dt_Customer before continuing.")
reference_date = df["Dt_Customer"].max()

df["Age"] = REFERENCE_YEAR - df["Year_Birth"]
df["Customer_Tenure_Days"] = (reference_date - df["Dt_Customer"]).dt.days
spending_columns = [
    "MntWines", "MntFruits", "MntMeatProducts",
    "MntFishProducts", "MntSweetProducts", "MntGoldProds"
]
df["Total_Spending"] = df[spending_columns].sum(axis=1)
df["Total_Children"] = df["Kidhome"] + df["Teenhome"]

# Consolidate related levels as in the original project notebook.
df["Education"] = df["Education"].replace({
    "Basic": "Undergraduate", "2n Cycle": "Undergraduate",
    "Graduation": "Graduate", "Master": "Postgraduate", "PhD": "Postgraduate"
})
df["Living_With"] = df["Marital_Status"].replace({
    "Married": "Partner", "Together": "Partner",
    "Single": "Alone", "Divorced": "Alone", "Widow": "Alone",
    "Absurd": "Alone", "YOLO": "Alone"
})

engineered_columns = ["Age", "Customer_Tenure_Days", "Total_Spending", "Total_Children"]
display(df[engineered_columns].describe().T)

## 5. Remove identifiers and redundant columns
`ID` is a row identifier, not a behavioural feature. Raw birth year, original marital status, household child-count components, enrollment date, and individual spending columns are removed after deriving `Age`, `Living_With`, `Total_Children`, `Customer_Tenure_Days`, and `Total_Spending`. This avoids representing the same information twice in the clustering features.

In [ ]:
columns_to_drop = [
    "ID", "Year_Birth", "Marital_Status", "Kidhome", "Teenhome", "Dt_Customer",
    *spending_columns
]
# Keep Response and Complain available for post-clustering descriptive checks only,
# but exclude them from the model feature matrix to keep clustering focused on profile inputs.
df_cleaned = df.drop(columns=columns_to_drop).copy()

print("Original shape:", df.shape)
print("After dropping redundant columns:", df_cleaned.shape)
display(df_cleaned.head())

## 6. Outlier inspection and treatment
Inspect selected feature distributions before applying the notebook’s explicit filters. The supplied approach removes records with `Age >= 90` and `Income >= 600,000`. Those cut-offs are retained here for traceability; they are project assumptions rather than universally valid statistical boundaries. The impact is reported. A future improvement would compare these rules with IQR/robust methods and domain review.

In [ ]:
outlier_features = ["Income", "Recency", "Age", "Total_Spending", "Total_Children"]
fig, axes = plt.subplots(len(outlier_features), 1, figsize=(10, 3 * len(outlier_features)))
for ax, feature in zip(axes, outlier_features):
    sns.boxplot(x=df_cleaned[feature], ax=ax)
    ax.set_title(f"Before outlier filtering — {feature}")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "outlier_boxplots_before.png", dpi=160, bbox_inches="tight")
plt.show()

rows_before_filters = len(df_cleaned)
age_removed = int((df_cleaned["Age"] >= 90).sum())
income_removed = int((df_cleaned["Income"] >= 600_000).sum())
# Apply sequential filters so row counts reflect actual pipeline behaviour.
df_cleaned = df_cleaned[df_cleaned["Age"] < 90].copy()
df_cleaned = df_cleaned[df_cleaned["Income"] < 600_000].copy()
rows_after_filters = len(df_cleaned)
print(f"Rows before filters: {rows_before_filters:,}")
print(f"Rows removed by age rule (initial count): {age_removed:,}")
print(f"Rows removed by income rule (initial count): {income_removed:,}")
print(f"Rows after filters: {rows_after_filters:,}")
print(f"Total rows removed by sequential filters: {rows_before_filters - rows_after_filters:,}")

## 7. Correlation heatmap
Use the numeric-feature heatmap to inspect relationships and possible redundancy. Correlation is descriptive, does not establish causation, and only reflects pairwise linear association.

In [ ]:
corr = df_cleaned.corr(numeric_only=True)
plt.figure(figsize=(13, 10))
sns.heatmap(corr, cmap="coolwarm", center=0, annot=True, fmt=".2f", annot_kws={"size": 7})
plt.title("SmartCart Numeric Feature Correlation Heatmap")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "numeric_correlation_heatmap.png", dpi=180, bbox_inches="tight")
plt.show()

# Show strongest non-diagonal absolute correlations for concise reporting.
upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
strong_pairs = upper.stack().rename("correlation").to_frame()
strong_pairs["absolute_correlation"] = strong_pairs["correlation"].abs()
strong_pairs = strong_pairs.sort_values("absolute_correlation", ascending=False)
display(strong_pairs.head(15))

## 8. Encode categorical variables
One-hot encoding converts `Education` and `Living_With` into numeric indicator columns. `handle_unknown="ignore"` makes the encoder more robust if additional categories appear in a future dataset.

In [ ]:
categorical_columns = ["Education", "Living_With"]
encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
encoded_array = encoder.fit_transform(df_cleaned[categorical_columns])
encoded_columns = encoder.get_feature_names_out(categorical_columns)
encoded_df = pd.DataFrame(encoded_array, columns=encoded_columns, index=df_cleaned.index)

# Response/Complain are kept separately for optional descriptive checks, not used by clustering.
post_cluster_fields = df_cleaned[["Response", "Complain"]].copy() if {"Response", "Complain"}.issubset(df_cleaned.columns) else None
feature_frame = pd.concat([
    df_cleaned.drop(columns=categorical_columns + [c for c in ["Response", "Complain"] if c in df_cleaned.columns]),
    encoded_df
], axis=1)
print("Encoded feature shape:", feature_frame.shape)
display(feature_frame.head())

## 9. Scale numerical features
StandardScaler puts features on a comparable scale, which matters for distance-based algorithms. Fit the scaler on the actual clustering features. The original units remain available in the profile data later.

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(feature_frame)
print("Scaled matrix shape:", X_scaled.shape)

## 10. PCA for dimensionality reduction and visualization
PCA creates a lower-dimensional representation. We use three components for 3D visualization and for reproducing the project’s intended workflow. Because dimensionality reduction can discard information, cluster selection should ideally be checked in the scaled feature space as a sensitivity analysis too.

In [ ]:
pca = PCA(n_components=3, random_state=RANDOM_STATE)
X_pca = pca.fit_transform(X_scaled)
explained_variance = pca.explained_variance_ratio_
print("Explained variance by PC:", np.round(explained_variance, 4))
print(f"Cumulative variance explained by first 3 PCs: {explained_variance.sum():.2%}")

fig = plt.figure(figsize=(9, 7))
ax = fig.add_subplot(111, projection="3d")
ax.scatter(X_pca[:, 0], X_pca[:, 1], X_pca[:, 2], alpha=0.65, s=18)
ax.set_xlabel("PC1"); ax.set_ylabel("PC2"); ax.set_zlabel("PC3")
ax.set_title("SmartCart Customers — PCA 3D Projection")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "pca_3d_projection.png", dpi=180, bbox_inches="tight")
plt.show()

## 11. Choose K: Elbow method and silhouette score
The Elbow method checks how within-cluster sum of squares (inertia) decreases as K grows. The silhouette score measures separation/cohesion for K ≥ 2. Neither method alone guarantees business-useful clusters; combine the curves with interpretability and stability checks.

In [ ]:
k_values = list(range(2, 11))
wcss = []
silhouette_scores = []
for k in range(1, 11):
    model = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
    model.fit(X_pca)
    wcss.append(model.inertia_)
    if k >= 2:
        silhouette_scores.append(silhouette_score(X_pca, model.labels_))

fig, ax1 = plt.subplots(figsize=(9, 5))
ax1.plot(range(1, 11), wcss, marker="o")
ax1.set_xlabel("Number of clusters (K)")
ax1.set_ylabel("WCSS / inertia")
ax1.set_title("Elbow Method")
ax1.set_xticks(range(1, 11))
plt.tight_layout()
plt.savefig(FIGURES_DIR / "elbow_method.png", dpi=160, bbox_inches="tight")
plt.show()

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(k_values, silhouette_scores, marker="o")
ax.set_xlabel("Number of clusters (K)")
ax.set_ylabel("Silhouette score")
ax.set_title("Silhouette Score by K")
ax.set_xticks(k_values)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "silhouette_scores.png", dpi=160, bbox_inches="tight")
plt.show()

selection_table = pd.DataFrame({"k": k_values, "silhouette_score": silhouette_scores})
selection_table["wcss"] = wcss[1:]
display(selection_table)
print("Highest silhouette K (diagnostic only):", int(selection_table.loc[selection_table["silhouette_score"].idxmax(), "k"]))

## 12. Fit clustering algorithms
The original notebook selected four clusters for both K-Means and Agglomerative Clustering. We preserve K=4 as the baseline for comparing the two algorithms, but also make K easy to change after reviewing the diagnostics above. Cluster IDs are arbitrary labels, not rankings.

Alongside the baseline comparison in PCA space, the notebook compares the same K and algorithms on the full standardized feature matrix. It also reports a small K-Means seed-consistency check. These are sensitivity diagnostics, not proof of business usefulness.

In [ ]:
CHOSEN_K = 4

kmeans = KMeans(n_clusters=CHOSEN_K, random_state=RANDOM_STATE, n_init=10)
labels_kmeans = kmeans.fit_predict(X_pca)

agglomerative = AgglomerativeClustering(n_clusters=CHOSEN_K, linkage="ward")
labels_agglomerative = agglomerative.fit_predict(X_pca)

def clustering_metrics(X_representation, labels, algorithm_name):
    return {
        "algorithm": algorithm_name,
        "clusters": int(len(np.unique(labels))),
        "silhouette": silhouette_score(X_representation, labels),
        "calinski_harabasz": calinski_harabasz_score(X_representation, labels),
        "davies_bouldin": davies_bouldin_score(X_representation, labels),
    }

metrics = pd.DataFrame([
    clustering_metrics(X_pca, labels_kmeans, "K-Means on PCA"),
    clustering_metrics(X_pca, labels_agglomerative, "Agglomerative on PCA"),
])
display(metrics)
metrics.to_csv(TABLES_DIR / "clustering_metrics.csv", index=False)

# Sensitivity analysis: compare clustering on PCA space with all standardized features.
sensitivity_rows = []
for representation_name, representation in [
    ("PCA (3 components)", X_pca),
    ("Full standardized features", X_scaled),
]:
    kmeans_labels = KMeans(n_clusters=CHOSEN_K, random_state=RANDOM_STATE, n_init=10).fit_predict(representation)
    agglomerative_labels = AgglomerativeClustering(n_clusters=CHOSEN_K, linkage="ward").fit_predict(representation)
    sensitivity_rows.append(clustering_metrics(representation, kmeans_labels, f"K-Means - {representation_name}"))
    sensitivity_rows.append(clustering_metrics(representation, agglomerative_labels, f"Agglomerative - {representation_name}"))

representation_sensitivity = pd.DataFrame(sensitivity_rows)
display(representation_sensitivity)
representation_sensitivity.to_csv(TABLES_DIR / "representation_sensitivity_metrics.csv", index=False)

# Compare K-Means partitions across a few random seeds; ARI=1 means identical partitions.
seed_labels = {
    seed: KMeans(n_clusters=CHOSEN_K, random_state=seed, n_init=10).fit_predict(X_pca)
    for seed in (42, 43, 44)
}
seed_stability = pd.DataFrame([
    {
        "reference_seed": 42,
        "comparison_seed": seed,
        "adjusted_rand_index": adjusted_rand_score(seed_labels[42], seed_labels[seed]),
    }
    for seed in (43, 44)
])
display(seed_stability)
seed_stability.to_csv(TABLES_DIR / "kmeans_seed_stability.csv", index=False)

## 13. Visualize cluster assignments
Visualizations show the first three principal components. Use these plots as a visual check, not proof that clusters are perfectly separated.

In [ ]:
def plot_clusters_3d(values, labels, title, filename):
    fig = plt.figure(figsize=(9, 7))
    ax = fig.add_subplot(111, projection="3d")
    scatter = ax.scatter(values[:, 0], values[:, 1], values[:, 2], c=labels, cmap="viridis", s=20, alpha=0.7)
    ax.set_xlabel("PC1"); ax.set_ylabel("PC2"); ax.set_zlabel("PC3")
    ax.set_title(title)
    fig.colorbar(scatter, ax=ax, pad=0.1, label="Cluster")
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / filename, dpi=180, bbox_inches="tight")
    plt.show()

plot_clusters_3d(X_pca, labels_kmeans, f"K-Means Customer Segments (K={CHOSEN_K})", "kmeans_clusters_pca.png")
plot_clusters_3d(X_pca, labels_agglomerative, f"Agglomerative Customer Segments (K={CHOSEN_K})", "agglomerative_clusters_pca.png")

## 14. Characterize and compare cluster sizes
For interpretation, use the original-scale customer features rather than scaled values. This section attaches labels generated by both algorithms and reports segment sizes. Model cluster numbers have no inherent order.

In [ ]:
profile_base = df_cleaned.drop(columns=[c for c in ["Response", "Complain"] if c in df_cleaned.columns]).copy()
profile_base["KMeans_Cluster"] = labels_kmeans
profile_base["Agglomerative_Cluster"] = labels_agglomerative

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.countplot(data=profile_base, x="KMeans_Cluster", hue="KMeans_Cluster", legend=False, ax=axes[0])
axes[0].set_title("Customer count by K-Means cluster")
sns.countplot(data=profile_base, x="Agglomerative_Cluster", hue="Agglomerative_Cluster", legend=False, ax=axes[1])
axes[1].set_title("Customer count by Agglomerative cluster")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "cluster_size_comparison.png", dpi=180, bbox_inches="tight")
plt.show()

cluster_counts = pd.DataFrame({
    "KMeans": profile_base["KMeans_Cluster"].value_counts().sort_index(),
    "Agglomerative": profile_base["Agglomerative_Cluster"].value_counts().sort_index(),
}).fillna(0).astype(int)
display(cluster_counts)
cluster_counts.to_csv(TABLES_DIR / "cluster_sizes.csv")

## 15. Cluster profile summary
Calculate the mean and median of interpretable variables by cluster. Medians help reduce the influence of skewed spending/income distributions. These summaries can support descriptive naming later (for example, “high-spend frequent shoppers”), but names should only be assigned after reviewing actual output values and should not imply causality.

In [ ]:
profile_features = [
    "Age", "Income", "Recency", "Total_Spending", "Total_Children",
    "Customer_Tenure_Days", "NumDealsPurchases", "NumWebPurchases",
    "NumCatalogPurchases", "NumStorePurchases", "NumWebVisitsMonth"
]
profile_features = [col for col in profile_features if col in profile_base.columns]

kmeans_mean_profile = profile_base.groupby("KMeans_Cluster")[profile_features].mean().round(2)
kmeans_median_profile = profile_base.groupby("KMeans_Cluster")[profile_features].median().round(2)
agg_mean_profile = profile_base.groupby("Agglomerative_Cluster")[profile_features].mean().round(2)

print("K-Means mean profile")
display(kmeans_mean_profile)
print("K-Means median profile")
display(kmeans_median_profile)
print("Agglomerative mean profile")
display(agg_mean_profile)

kmeans_mean_profile.to_csv(TABLES_DIR / "kmeans_cluster_mean_profile.csv")
kmeans_median_profile.to_csv(TABLES_DIR / "kmeans_cluster_median_profile.csv")
agg_mean_profile.to_csv(TABLES_DIR / "agglomerative_cluster_mean_profile.csv")

## 16. Income and spending pattern by segment

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
sns.scatterplot(data=profile_base, x="Total_Spending", y="Income", hue="KMeans_Cluster", palette="tab10", alpha=0.75, ax=axes[0])
axes[0].set_title("Income vs total spending — K-Means")
sns.scatterplot(data=profile_base, x="Total_Spending", y="Income", hue="Agglomerative_Cluster", palette="tab10", alpha=0.75, ax=axes[1])
axes[1].set_title("Income vs total spending — Agglomerative")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "income_spending_by_cluster.png", dpi=180, bbox_inches="tight")
plt.show()

## 17. Optional post-clustering response/complaint audit
These fields are excluded from the clustering features. Reviewing them after segmentation can help form hypotheses about how segments differ; it must not be presented as evidence of model accuracy, causation, or future response prediction.

In [ ]:
if post_cluster_fields is not None:
    audit = post_cluster_fields.copy()
    audit["KMeans_Cluster"] = labels_kmeans
    response_rates = audit.groupby("KMeans_Cluster")["Response"].mean().rename("response_rate")
    complaint_rates = audit.groupby("KMeans_Cluster")["Complain"].mean().rename("complaint_rate")
    engagement_audit = pd.concat([response_rates, complaint_rates], axis=1).round(3)
    display(engagement_audit)
    engagement_audit.to_csv(TABLES_DIR / "post_cluster_response_complaint_audit.csv")
else:
    print("Response/Complain fields were not present; skipping the optional audit.")

## 18. Save customer-level cluster assignments and final checks

The CSV stores the cleaned interpretable fields plus both algorithms’ cluster labels. It intentionally does not include the original `ID` identifier in this export. Since clustering is exploratory, validate the profiles with stakeholders and monitor whether actions based on them have the intended effect before using segments operationally.

In [ ]:
assignments = profile_base.copy()
assignments.to_csv(TABLES_DIR / "customer_cluster_assignments.csv", index=False)

print("Final checks")
print(f"Customers retained: {len(assignments):,}")
print(f"K-Means clusters: {assignments['KMeans_Cluster'].nunique()}")
print(f"Agglomerative clusters: {assignments['Agglomerative_Cluster'].nunique()}")
print("Missing values in model feature matrix:", int(feature_frame.isna().sum().sum()))
print("Exports saved to:", TABLES_DIR.resolve())

## 19. Conclusion and next steps

This project builds an end-to-end unsupervised segmentation workflow: data quality checks, missing-value treatment, feature engineering, filtering, correlation analysis, one-hot encoding, scaling, PCA, K selection diagnostics, two clustering algorithms, and customer-profile summaries.

**Before presenting results:**
- Run the notebook from top to bottom and check the saved tables.
- Record the actual best silhouette score, chosen K rationale, cluster sizes, and profile differences in the report.
- Re-check whether PCA-based clustering is preferable to clustering directly on standardized features.
- Test cluster stability across random seeds or resampled data.
- Avoid describing a cluster as “churn-prone” unless churn outcomes and a suitable validation design support that claim.